In [ ]:
tree = tskit.load("Remapped/Chr" + str(chromosome) + ".trees")
treePos = [x.position for x in tree.sites()]
chrSample = pd.read_table("All_snps_with_outgroup_info_newAA/Chr" + str(chromosome) + "_relate.sample")


# %%
meta = pd.read_csv("TsinferMetaData_Vcf.txt")
meta.head()

sampleMeta = chrSample.merge(meta, how = "left", left_on = "ID_1", right_on = "ID")
sampleMeta = sampleMeta.iloc[1: , :]

complex_schema = tskit.MetadataSchema({
    'codec': 'json',
    'additionalProperties': False,
    'properties': {'name': {'description': 'ENA accession number',
                                 'type': 'string'},
                   'pop': {'description': 'Lineage',
                           'type': 'string'},
                    'subpop': {'description': 'Subspecies',
                           'type': 'string'}},
    'required': ['name', 'pop', 'subpop'],
    'type': 'object',
})


# Add meta data
tables = tree.dump_tables()
tables.individuals.metadata_schema = complex_schema



# Add metadata
metadata_column = [{"name": list(sampleMeta.ID_1)[ind], "pop": list(sampleMeta.Pop)[ind], "subpop": list(sampleMeta.SubPop)[ind]} for ind in range(tree.num_individuals)]
encoded_metadata_column = [
    tables.individuals.metadata_schema.validate_and_encode_row(r) for r in metadata_column
]
md, md_offset = tskit.pack_bytes(encoded_metadata_column)
tables.individuals.set_columns(flags=range(tree.num_individuals), metadata=md, metadata_offset=md_offset)

tree_meta = tables.tree_sequence()

samples_listed_by_all = [tree.samples() for ind in tree.samples()]
gnn = tree.genealogical_nearest_neighbours(
    tree.samples(), samples_listed_by_all
)

sample_ids = [n.id for n in tree.individuals()]
sample_names = [
    tree_meta.individual(n).metadata["name"]
    for n in sample_ids
]